# R2 行动研究 · 牛津 Tutorial LLM 仿真 (Oxford + HBS + Hattie)

## Persona Prompt (tutor 侧)

You are an Oxford tutorial fellow in **行动研究 (Action Research: participatory cycles, diagnose-act-observe-reflect, MIT Sloan action learning)**. You run one-on-one tutorials in the style of Oxford PPE / Oxford MBA.

**Tutoring rules (strict, never violate):**
1. **Never give direct answers.** You are a Socratic interrogator, not a lecturer. 不直接给答案。
2. **Use Socratic questioning** exclusively: every turn ends with a probing question.
3. **Act as HBS devil's advocate**: stress-test the student's claims the way a HBS case-method professor would - "what's the counterfactual?", "where's the evidence?", "who disagrees?".
4. **Reject vague claims.** If the student says "AR is iterative" without specifying Plan-Act-Observe-Reflect or naming the round count, you reject and re-prompt.
5. **End each turn with a probing question** drawn from: 为什么 (why) / 反例 (counterexample) / 若前提变 (what if premise changes) / 凭什么 (on what grounds) / 如何 (how).
6. **Limit frequency**: 1 tutorial per day per unit (限频, 防依赖, prevent tutor dependency). The student must struggle alone between tutorials.
7. **Track student_model.json**: after each turn, update mastery (0-1) and blind_spots list.

> 牛津 tutorial 的核心不是"教", 而是"逼学生思考"。每次回答必须以追问结尾。


## Pre-Tutorial Task (强制 retrieval, 上交后才能进 tutorial)

> 牛津 tutorial 的契约: **学生先写, tutor 后问**。不提交 pre-tutorial task, tutorial 不开始。

**任务 (任选一, 300-500 字, 提交到 `pre_tutorial_essay.txt`):**

### 选项 A (认识论层)
Susman & Evered (1978) 称行动研究者为 "change agent", 而 Yin (2018) 案例研究的研究者尽量不干预。请论证: **如果你博士论文研究"企业营销 AI 转型的迭代部署", 你应选 AR 还是 Case Study? 给 3 个理由。**

### 选项 B (循环建模层)
给定 4 轮 Plan-Act-Observe-Reflect 螺旋, 每轮收集 4 个 KPI (决策时间/方案质量/AI使用率/团队满意度)。请写出 pandas DataFrame schema (列名 + dtypes), 并说明如何用它识别"高杠杆轮次"。

### 选项 C (贝叶斯层)
设先验 Beta(2, 2), 每轮观察 (successes, failures)。请推导第 3 轮后验 α' 和 β' 的公式, 并解释后验均值如何随观察数据变化。

---

**提交格式:**
```python
essay = "[你的 300-500 字 essay 写在这里, 必须引用 notes.md 的具体概念]"
open("pre_tutorial_essay.txt", "w").write(essay)
```


In [ ]:
# Socratic multi-turn loop (>=4 轮, 静态 if/else 模拟 Socratic 追问, 不真调 API)
# 本 cell 用静态规则模拟牛津 tutor 的追问, 不调用 openai/anthropic
# 每轮必含一个苏格拉底问: 为什么/反例/若前提变/凭什么/如何

import json, os

STUDENT_ESSAY = open("pre_tutorial_essay.txt", "r").read() if os.path.exists("pre_tutorial_essay.txt") else ""

# 苏格拉底问题库 (>=5 个, 领域特定)
SOCRATIC_QUESTIONS = [
    "为什么 AR 用 trustworthiness 而非 internal validity? 凭什么 Lincoln & Guba 的四准则更适合干预式研究?",
    "若你的 Beta(2,2) 先验变成 Beta(10,2), 后验会偏向哪一边? 这对'干预有效'的结论有何影响?",
    "举一个反例: 什么场景下行动研究比 DSR 更差? 即什么时候不该用 AR?",
    "如何用天道推演的'沙盘模拟 3 层'能力, 在 Plan 阶段预演 3 个干预方案? 各推演什么 immediate/near/far 未来走向?",
    "凭什么说 Round 2 是高杠杆轮次? 你的判断依据是改善幅度 > 20% 还是别的阈值? 为什么是这个阈值?",
    "为什么 PAR 的共创度公式是 (Key Players + Show Consideration) / 总数, 而不含 Meet Their Needs? 反例: 一个高权力低利益的利益相关方被排除在共创度外合理吗?",
]

# 静态模拟 4 轮 Socratic 对话
# 每轮: tutor 问 -> 学生答 (静态模拟) -> tutor 评估 + 下一问
dialogue_log = []

def tutor_turn(round_idx, student_answer):
    """静态 if/else 模拟牛津 tutor 的 Socratic 追问"""
    round_idx = round_idx % len(SOCRATIC_QUESTIONS)
    question = SOCRATIC_QUESTIONS[round_idx]

    # 静态评估学生答案 (基于关键词匹配, 不调 LLM)
    answer_lower = student_answer.lower()
    if "trustworthiness" in answer_lower or "可信度" in student_answer:
        assessment = "[TASK] 你命中了 trustworthiness 关键词, 但还没说清 Credibility vs Confirmability 的量化差异。"
        follow_up = "追问: Credibility 用三角验证 (数据源数) 量化, Confirmability 用成员校验率量化 - 为什么这两个不能互换?"
    elif "beta" in answer_lower and ("alpha" in answer_lower or "α" in student_answer):
        assessment = "[PROCESS] 你正确引用了 Beta 更新, 但没解释共轭性。"
        follow_up = "追问: Beta-Binomial 共轭意味着什么? 为什么先验和后验都是 Beta 分布?"
    elif "plan" in answer_lower or "规划" in student_answer or "沙盘" in student_answer:
        assessment = "[SELF-REG] 你提到 Plan 阶段, 但没说清天道推演的 5 项能力如何映射。"
        follow_up = "追问: 天道推演的'局势感知'对应 AR 的哪个阶段? '概率评估'又对应哪个?"
    elif "par" in answer_lower or "共创" in student_answer or "权力" in student_answer:
        assessment = "[FEED-FORWARD] 你提到了 PAR, 但共创度公式没写全。"
        follow_up = "追问: 如果某轮 Key Players=2, Show Consideration=1, 总利益相关方=5, 共创度是多少? 这个值合格吗?"
    else:
        assessment = "[TASK] 你的回答太模糊, 拒绝接受。必须引用 notes.md 的具体概念 (Plan-Act-Observe-Reflect / trustworthiness / Beta 共轭 / 天道推演 5 能力)。"
        follow_up = question

    dialogue_log.append({
        "round": round_idx,
        "tutor_question": question,
        "student_answer": student_answer[:200],
        "tutor_assessment": assessment,
        "tutor_follow_up": follow_up,
    })
    return follow_up

# 模拟 4 轮对话 (静态学生答案, 用于演示 Socratic loop 结构)
mock_student_answers = [
    "AR 用 trustworthiness 因为研究者是干预者, 不能用旁观者的 internal validity",
    "Beta(2,2) 先验, 第3轮后验 alpha=2+successes, beta=2+failures, 共轭因为都是 Beta",
    "Plan 阶段用天道推演的沙盘模拟, 推演 3 个干预方案的 immediate/near/far 未来",
    "PAR 共创度 = (Key Players + Show Consideration) / 总数, Round 2 共创度=0.6",
]

print("=" * 70)
print("牛津 Tutorial R2 行动研究 · Socratic Loop (4 轮静态模拟)")
print("=" * 70)
for i, ans in enumerate(mock_student_answers):
    next_q = tutor_turn(i, ans)
    print(f"\n--- Round {i+1} ---")
    print(f"Tutor 问: {SOCRATIC_QUESTIONS[i]}")
    print(f"学生答: {ans}")
    print(f"Tutor 评估: {dialogue_log[-1]['tutor_assessment']}")
    print(f"Tutor 追问: {next_q}")

print(f"\n{'='*70}")
print(f"对话记录已存入 dialogue_log ({len(dialogue_log)} 轮)")
print(f"苏格拉底问总数: {len(SOCRATIC_QUESTIONS)} (>=5)")


In [ ]:
# student_model.json 读写: 记录掌握度 + 盲点
import json, os

STUDENT_MODEL_PATH = "student_model.json"

def load_student_model():
    if os.path.exists(STUDENT_MODEL_PATH):
        with open(STUDENT_MODEL_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {
        "unit": "U-R2",
        "topic": "行动研究 (Action Research)",
        "mastery": {
            "ILO1_认识论": 0.0,
            "ILO2_pandas循环建模": 0.0,
            "ILO3_效度量化": 0.0,
            "ILO4_PAR共创": 0.0,
            "ILO5_贝叶斯更新": 0.0,
            "ILO6_天道推演预演": 0.0,
        },
        "blind_spots": [],
        "tutorial_count_today": 0,
        "last_tutorial_date": None,
    }

def update_mastery(model, ilo_key, delta):
    """根据 tutor 评估更新掌握度 (0-1)"""
    old = model["mastery"].get(ilo_key, 0.0)
    new = max(0.0, min(1.0, old + delta))
    model["mastery"][ilo_key] = round(new, 3)
    return new

def add_blind_spot(model, spot):
    if spot not in model["blind_spots"]:
        model["blind_spots"].append(spot)

def save_student_model(model):
    with open(STUDENT_MODEL_PATH, "w", encoding="utf-8") as f:
        json.dump(model, f, ensure_ascii=False, indent=2)

# 演示: 基于 cell3 的 dialogue_log 更新 student_model
model = load_student_model()

ilo_deltas = [
    ("ILO1_认识论", +0.15, "Credibility vs Confirmability 量化差异未说清"),
    ("ILO5_贝叶斯更新", +0.20, "Beta-Binomial 共轭性解释不足"),
    ("ILO6_天道推演预演", +0.25, "天道推演 5 能力映射不全"),
    ("ILO4_PAR共创", +0.30, "共创度公式应用正确但阈值判断缺失"),
]

for ilo_key, delta, blind in ilo_deltas:
    new_mastery = update_mastery(model, ilo_key, delta)
    if delta < 0.25:
        add_blind_spot(model, blind)

save_student_model(model)

print("=" * 70)
print("student_model.json 已更新")
print("=" * 70)
print(json.dumps(model, ensure_ascii=False, indent=2))


## Hattie (2007) 四级 Formative Feedback

> Hattie & Timperley (2007) 的 4 级反馈模型。**避免 Self 级表扬** (如"你真聪明") - Self 级反馈与学习成效负相关。本 tutorial 只用 TASK / PROCESS / SELF-REG / FEED-FORWARD 四级。

### [TASK] 任务级反馈 - 针对具体任务答案的正确性
- **示例 (本单元)**: "你在 pre-tutorial essay 中说'AR 是迭代的', 这正确但太模糊。必须明确说 4 轮 Plan-Act-Observe-Reflect 螺旋, 而非泛泛的'迭代'。"
- **作用**: 修正具体错误, 但不涉及学习策略。

### [PROCESS] 过程级反馈 - 针对学习策略/方法
- **示例 (本单元)**: "你试图用频率派 p-hat 评估干预有效性, 但 AR 的每轮观察应该用 Beta-Binomial 共轭更新后验。策略错了, 不是答案错。改用 scipy.stats.beta 顺序更新。"
- **作用**: 教学生"如何学", 而非"学什么"。

### [SELF-REG] 自我调节级反馈 - 针对元认知/自我监控
- **示例 (本单元)**: "你在 Round 1 触发了 weak_loop, 但你没有回退到 practice.md D4 阶段 1 重新看 worked example。你的自我监控缺失 - 你需要学会识别'我卡住了, 该回退'的信号。"
- **作用**: 培养元认知, 让学生自己诊断盲点。

### [FEED-FORWARD] 前馈级反馈 - 针对下一步方向
- **示例 (本单元)**: "你的 ILO5 (贝叶斯) 掌握度 0.45, 低于 mastery 阈值 0.80。下一步: (1) 重做 practice.md D4 阶段 1-3; (2) 读 notes.md § 贝叶斯更新与行动研究; (3) 24h 后再约一次 tutorial (限频 1 次/天)。"
- **作用**: 给出具体的下一行动路径, 连接到 practice.md 弱项循环。

> **禁止 Self 级表扬**: 不说"你真聪明"/"做得好" - Hattie 元分析显示 Self 级反馈 effect size 仅 0.14 (低于 0.40 的 hinge point), 甚至可能负向。本 tutorial 的反馈全部聚焦 TASK/PROCESS/SELF-REG/FEED-FORWARD 四级。


## 限频与 Exit Artifact

### 限频 (防依赖)
- **每单元 1 次/天**: 本 tutorial 每天最多 1 次对话。student_model.json 的 `tutorial_count_today` 字段记录当日次数。
- **为什么限频**: 牛津 tutorial 的价值在于"逼学生独自挣扎"。每天 1 次足够 - 剩下 23 小时学生必须自己读 notes.md / 做 practice.md / 跑 starter.ipynb。过度依赖 tutor 会削弱自我调节能力 (Hattie SELF-REG 级反馈失效)。
- **豁免**: 连续 2 次触发 weak_loop 后, 可申请 1 次额外 tutorial (1-on-1 深度复盘)。

### Exit Artifact (本 tutorial 结束时必须产出)

> Tutorial 结束不是"老师说你走了", 而是"学生产出 exit artifact"。无 exit artifact = tutorial 未完成。

**Exit artifact 必含:**

1. **2-3 个盲点** (从 student_model.json 的 `blind_spots` 字段提取, 学生自己写一句话总结每个盲点):
   - 盲点 1: _________________________________________________
   - 盲点 2: _________________________________________________
   - 盲点 3: _________________________________________________

2. **推荐复习单元** (基于盲点, 学生自己选):
   - 若盲点含"认识论混淆" -> 复习 R1 (设计科学研究) + R2 § 关键回顾 1/5
   - 若盲点含"循环建模" -> 复习 practice.md D1+D2 + starter.ipynb TODO1+2
   - 若盲点含"效度量化" -> 复习 practice.md D3 + notes.md § 关键回顾 4
   - 若盲点含"贝叶斯" -> 复习 practice.md D4 + notes.md § 贝叶斯更新
   - 若盲点含"天道推演" -> 复习 notes.md § 天道推演预演 + CLAUDE.md 天道推演系统
   - 若盲点含"PAR 共创" -> 复习 practice.md D5 + notes.md § 关键回顾 3

3. **下次 tutorial 的 pre-tutorial task 选项** (学生承诺 24h 内完成):
   - [ ] 选项 A (认识论层) - 重写 AR vs Case Study 论证, 含 3 个具体理由
   - [ ] 选项 B (循环建模层) - 重写 pandas schema, 含 4 轮 × 4 KPI + 高杠杆轮次识别
   - [ ] 选项 C (贝叶斯层) - 重推 Beta(2,2) 共轭更新, 含 95% HDI 计算

---

**Tutor 签字**: 本 tutorial 已完成, exit artifact 已存档。下次 tutorial 限 24h 后。
